# CSV Attachment To Data Table

Run this notebook from the File Analysis interface on a CSV file attached to a test result. It creates a SystemLink DataFrame table and associates it to that same result.

In [ ]:
file_ids = []
test_result_id = ""

In [ ]:
import io
import math
import os
from pathlib import Path

import pandas as pd
import requests
import scrapbook as sb

api_key = os.getenv("SYSTEMLINK_API_KEY")
systemlink_uri = os.getenv("SYSTEMLINK_HTTP_URI", "").rstrip("/")
session = requests.Session()
session.headers.update({"x-ni-api-key": api_key, "accept": "application/json"})
file_base = f"{systemlink_uri}/nifile/v1/service-groups/Default/files"
testmonitor_base = f"{systemlink_uri}/nitestmonitor/v2"
dataframe_base = f"{systemlink_uri}/nidataframe/v1"

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def api_json(method, url, **kwargs):
    response = session.request(method, url, timeout=60, **kwargs)
    response.raise_for_status()
    if response.status_code == 204 or not response.content:
        return None
    return response.json()

def file_metadata(file_id):
    return api_json("GET", f"{file_base}/{file_id}")

def file_bytes(file_id):
    response = session.get(f"{file_base}/{file_id}/data", timeout=60)
    response.raise_for_status()
    return response.content

def parse_results(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("results", "items", "data"):
            value = payload.get(key)
            if isinstance(value, list):
                return value
    return []

def result_for_file(file_id):
    payload = api_json("POST", f"{testmonitor_base}/query-results", json={"filter": "fileIds.Contains(@0)", "substitutions": [file_id], "take": 2})
    results = parse_results(payload)
    require(len(results) == 1, f"Expected exactly one test result for file {file_id}, found {len(results)}.")
    return results[0]

def infer_type(series):
    non_null = series.dropna()
    if not non_null.empty and pd.api.types.is_object_dtype(series):
        parsed = pd.to_datetime(non_null, utc=True, errors="coerce")
        if parsed.notna().all():
            return "TIMESTAMP"
    if pd.api.types.is_bool_dtype(series):
        return "BOOL"
    if pd.api.types.is_integer_dtype(series):
        return "INT64"
    if pd.api.types.is_float_dtype(series):
        return "FLOAT64"
    return "STRING"

def serialize(value, data_type):
    if pd.isna(value):
        return None
    if data_type == "BOOL":
        return "true" if bool(value) else "false"
    if data_type == "INT64":
        return str(int(value))
    if data_type == "FLOAT64":
        numeric = float(value)
        if math.isnan(numeric):
            return "NaN"
        if math.isinf(numeric):
            return "Infinity" if numeric > 0 else "-Infinity"
        return format(numeric, ".17g")
    if data_type == "TIMESTAMP":
        ts = pd.Timestamp(value)
        if ts.tzinfo is None:
            ts = ts.tz_localize("UTC")
        else:
            ts = ts.tz_convert("UTC")
        return ts.isoformat(timespec="milliseconds").replace("+00:00", "Z")
    return str(value)

def query_existing_tables(result_id, source_file_id):
    payload = api_json("POST", f"{dataframe_base}/query-tables", json={"filter": "testResultId == @0 and properties[\"sourceFileId\"] == @1", "substitutions": [result_id, source_file_id], "take": 100})
    return payload.get("tables", []) if payload else []

def create_table(dataframe, result_id, source_file_id, source_file_name, workspace_id):
    columns = [{"name": "rowIndex", "dataType": "INT64", "columnType": "INDEX"}]
    data_types = {"rowIndex": "INT64"}
    converted = dataframe.convert_dtypes()
    for name in converted.columns:
        data_type = infer_type(converted[name])
        definition = {"name": name, "dataType": data_type}
        if converted[name].isna().any():
            definition["columnType"] = "NULLABLE"
        columns.append(definition)
        data_types[name] = data_type

    existing = query_existing_tables(result_id, source_file_id)
    if existing:
        api_json("POST", f"{dataframe_base}/delete-tables", json={"ids": [table["id"] for table in existing]})

    created = api_json("POST", f"{dataframe_base}/tables", json={"name": f"{Path(source_file_name).stem} Data Table", "columns": columns, "testResultId": result_id, "workspace": workspace_id, "properties": {"sourceFileId": source_file_id, "sourceFileName": source_file_name, "sourceType": "csv", "generatedBy": "csv-attachment-to-datatable"}})
    table_id = created["id"]

    rows = []
    for row_index, row in enumerate(converted.itertuples(index=False, name=None)):
        serialized_row = [str(row_index)]
        for column_name, value in zip(converted.columns, row):
            serialized_row.append(serialize(value, data_types[column_name]))
        rows.append(serialized_row)

    api_json("POST", f"{dataframe_base}/tables/{table_id}/data", json={"frame": {"columns": ["rowIndex", *list(converted.columns)], "data": rows}, "endOfData": True})
    return table_id, converted

require(api_key, "SYSTEMLINK_API_KEY is not set.")
require(systemlink_uri, "SYSTEMLINK_HTTP_URI is not set.")
require(len(file_ids) == 1, "Select exactly one CSV file.")
selected_file_id = file_ids[0]
metadata = file_metadata(selected_file_id)
file_name = metadata.get("properties", {}).get("Name", selected_file_id)
require(file_name.lower().endswith(".csv"), f"Expected a CSV file, received {file_name}.")
result = {"id": test_result_id.strip()} if test_result_id.strip() else result_for_file(selected_file_id)
workspace_id = metadata.get("workspace") or result.get("workspace")
require(workspace_id, "Unable to determine workspace.")
csv_df = pd.read_csv(io.BytesIO(file_bytes(selected_file_id)))
require(not csv_df.empty, "CSV file has no rows.")
created_table_id, preview_df = create_table(csv_df, result["id"], selected_file_id, file_name, workspace_id)

In [ ]:
preview_head = preview_df.head(25)
result = [
    {
        "display_name": "Created Table Id",
        "id": "created_table_id",
        "type": "string",
        "data": created_table_id
    },
    {
        "display_name": "Preview",
        "id": "preview",
        "type": "data_frame",
        "data": {
            "columns": list(preview_head.columns),
            "values": preview_head.astype(object).where(pd.notnull(preview_head), None).values.tolist()
        }
    }
]
sb.glue("result", result)